# Part 1 · How PedNStream works
### From one corridor to a city: walking, OD weights, and a network of your own

In about 40 minutes you will
1. see the **one rule** PedNStream uses for walking,
2. watch a **queue** grow in a single corridor,
3. find out why **who goes where** (the OD weights) matters as much as **how many** people there are,
4. run the same model on **real Amsterdam streets**,
5. **build your own network**, with your own OD weights and arrival pattern.

Work in pairs: one person clicks, the other takes notes.

**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

## ▶ Step 1: install (run this first, ≈30 s)
This is how you install **any** Python package from GitHub. `pednstream-workshop` is a small
package with this workshop's network and plotting helpers; installing it also installs
**PedNStream** itself. If Colab restarted, run it again.
You can ignore the note *"you may need to restart the kernel"*.

In [ ]:
%pip install -q "pednstream-workshop @ git+https://github.com/WaimenMak/pednstream-workshop.git"
import json, importlib.metadata as md
if json.loads(md.distribution("pednstream").read_text("direct_url.json") or "{}").get("vcs_info", {}).get("commit_id") != "4a85236b3c47b17f25c2bfe8d5c0ef35651900a7":
    %pip install -q --force-reinstall --no-deps "pednstream @ git+https://github.com/WaimenMak/PedNStream.git@4a85236b3c47b17f25c2bfe8d5c0ef35651900a7"

import matplotlib.pyplot as plt
import pednstream_workshop
if tuple(map(int, pednstream_workshop.__version__.split("."))) < (1, 12, 0):
    raise RuntimeError("An old version of the workshop package is loaded. "
                       "Click Runtime → Restart session, then run this cell again.")
from pednstream_workshop import (run, compare, compare_maps, make_network, arrival_plan,
                                 plot_speed_density, AMSTERDAM)
plt.rcParams["figure.dpi"] = 90
print("✅ Ready! Scroll down and continue.")

## 1 · PedNStream in one minute
PedNStream does not move individual dots around. It keeps track of **how many people** are on each
corridor and updates that every **time step** (here 10 seconds). That makes it fast: a whole city
centre runs in seconds.

**The network** is a set of **nodes** (gates, junctions, destinations) joined by **corridors** (links).
Every corridor has a length and a width, and people can walk it both ways.

**Every 10 seconds, four things happen:**

| | what happens | decided by | you set it with |
|---|---|---|---|
| 1 | people **arrive** at the origins (gates) | the **arrival pattern**: people per minute | `arrivals=` |
| 2 | each new person gets a **destination** and keeps it | the **OD weights** | `od=` |
| 3 | at each junction people pick their **next corridor** | **route choice**: shorter, wider and emptier routes are more attractive | `route_choice=` |
| 4 | people **walk** | how crowded the corridor is (below) | corridor `width` |

When the next corridor is full, people cannot step into it: they wait, and the queue grows
**backwards** into the corridors behind them (*spill-back*). If even the first corridor at a gate is full,
new arrivals are **turned away**.

PedNStream is built on the *Link Transmission Model* from traffic engineering, adapted to people walking
in both directions. Press ▶ to see its walking rule.

In [ ]:
#@title ▶ The walking rule {display-mode: "form"}
corridor_width_m = 2  #@param {type:"slider", min:0.5, max:6, step:0.5}
plot_speed_density(width=corridor_width_m); plt.show()

**How to read this**
* Up to **2 people per m²** everyone walks at full speed (1.1 m/s). Above that people slow down, and at
  **6 per m²** nobody moves.
* Flow = density × speed. It is highest at 2 per m²: that is the corridor's **capacity**, and it grows
  with the width. A wider corridor passes more people per minute; a longer one does not.
* Density counts **both directions**: people walking towards you take up room too.

> **Try:** set the width to 1 m, then 3 m. How many people per minute can each one pass?

<details><summary>💡 Our answer</summary>
1 m: about <b>130</b> people per minute. 3 m: about <b>400</b>. Three times as wide passes three times as many
people. (Capacity = 2 people per m² × 1.1 m/s × width × 60 s.)
</details>

---
## 2 · First run: one corridor with a bottleneck
```
   0 ─────── 1 ─────── 2 ─────── 3 ═══════ 4 ─────── 5
Entrance          40 m each, 3 m wide      ↑          Exit
                                       bottleneck
```
For 20 minutes people arrive at the entrance, then nobody. Everyone walks to the exit. Press ▶, then change
the bottleneck width and press ▶ again.

In [ ]:
#@title ▶ Run the corridor {display-mode: "form"}
bottleneck_width_m = 1.5  #@param {type:"slider", min:0.5, max:3, step:0.5}
people_per_minute = 200  #@param {type:"slider", min:20, max:400, step:20}

corridor = make_network(
    corridors=["0-1", "1-2", "2-3", "3-4", "4-5"],
    positions={0: (0, 0), 1: (40, 0), 2: (80, 0), 3: (120, 0), 4: (160, 0), 5: (200, 0)},  # metres
    origins=[0], destinations=[5],
    widths={"3-4": bottleneck_width_m},
    names={0: "Entrance", 5: "Exit"},
    folder="corridor", minutes=40)
rush = arrival_plan([people_per_minute, people_per_minute], block_minutes=10)   # 2 x 10 minutes, then nobody
one = run(network_dir=corridor, arrivals={0: rush})
one.dashboard("One corridor with a bottleneck", flow=True); plt.show()
display(one.kpis().to_frame("value"))

**How to read the dashboard** (you'll see it for every network)
* **Map** (left): corridors coloured by density at the busiest moment: green is free walking, yellow and
  red is crowded.
* **Density per corridor** (top right): one line per corridor. The dashed line is the 2 per m² where
  people start to slow down.
* **In and out** (bottom right): blue is people entering (your arrival pattern), red is people reaching a
  destination, the dashed line is how many people are inside.

**Discuss with your partner** (bottleneck 1.5 m, 200 people per minute)
1. Which corridor gets crowded **first**? Which one **next**?
2. About how many people per minute reach the exit while there is a queue? More or fewer than the 200 who arrive?
3. Make the bottleneck **2 m**, then **0.5 m**. What changes?

<details><summary>💡 Our answers</summary>
<b>1.</b> First the bottleneck itself, 3_4 (minute 3), then the corridor just before it, 2_3 (minute 5), then
1_2 (minute 14). The queue grows <b>backwards</b>, towards the entrance. The corridor after the bottleneck,
4_5, stays quiet.<br><br>
<b>2.</b> About <b>150</b> per minute, fewer than the 200 who arrive, so the queue keeps growing until the arrivals
stop at minute 20. It is even fewer than the ≈200 a 1.5 m corridor can pass at best: the bottleneck is packed
(about 3 people per m²), and a packed corridor moves slowly. That is the right-hand side of the flow curve.<br><br>
<b>3.</b> 2 m: no queue. A 2 m corridor passes up to ≈260 people per minute, more than the 200 who arrive.
0.5 m: the queue reaches the entrance after about 10 minutes, and about 1,100 of the 4,000 people are
turned away.
</details>

In [ ]:
#@title 🎬 Animate the corridor (optional, ≈15 s) {display-mode: "form"}
display(one.animate(every=3))

---
## 3 · Who goes where? Meet *Station Square*
```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```
* **Two entrances**: West Gate **A** enters at the Plaza, South Gate **B** enters at the Hall.
* **Two destinations**: the **Platform X** and the **City Exit Y**.
* Corridor **2_3** (Plaza–Hall) is narrow: only **2 m** wide.
* A rush of people arrives at both gates, peaking around minute 17. The rush is **the same in every run**.
* Where there are two routes, people choose by **distance**, **crowding** and **width**.

## 4 · What is an OD weight?
An **OD weight** says how strongly the people entering at a gate (the *origin*) are drawn to each
*destination*. Weights are **relative**: `3` and `1` mean "three times as many head to the Platform as
to the City Exit". The arrival pattern sets how *many* people enter; the weights only decide where they head.

The cell below opens **only Gate A**. Press ▶, then move the sliders and press ▶ again.

In [ ]:
#@title ▶ Try OD weights for Gate A {display-mode: "form"}
weight_A_to_Platform = 3  #@param {type:"slider", min:0, max:10, step:1}
weight_A_to_CityExit = 1  #@param {type:"slider", min:0, max:10, step:1}
only_A = run(od={"0_5": weight_A_to_Platform, "0_6": weight_A_to_CityExit}, origins=[0])

od = only_A.network.od_tracker.od_table()        # PedNStream follows every person from gate to destination
to_x, to_y = od[(0, 5)]["entered"], od[(0, 6)]["entered"]
share = to_x / max(to_x + to_y, 1e-9)
print(f"Gate A sent {share:.0%} to Platform X ({to_x:.0f} people) "
      f"and {1 - share:.0%} to City Exit Y ({to_y:.0f} people)")

> **Try:** 1 & 1, then 10 & 10, then 1 & 0. What share goes to the Platform each time?
>
> Every person who enters gets a destination drawn from the weights and keeps it all the way.
> **The weights decide where people go; the simulation decides which route they take and when they arrive.**

<details><summary>💡 Our answer</summary>
1 & 1: <b>50%</b>. 10 & 10: also <b>50%</b>, because only the ratio counts, not the size. 1 & 0: <b>100%</b>, nobody
goes to the City Exit. (3 & 1 gives 75%.) The number of people entering is the same every time: the weights
only split them.
</details>

---
## 5 · Same totals, different mix ⭐
Now both gates are open. Three ways to divide the travellers, and in all three **half of all travellers
go to each destination**. Only *who* goes where differs:

| scenario | Gate A goes to | Gate B goes to |
|---|---|---|
| **baseline** | half X, half Y | half X, half Y |
| **crossing** | Platform X (east) | City Exit Y (north) |
| **parallel** | City Exit Y (north) | Platform X (east) |

Guess first with your partner: will the three behave the same? Then press ▶.

In [ ]:
#@title ▶ Run the three scenarios {display-mode: "form"}
base = run()
crossing = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1})
parallel = run(od={"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0})
runs = {"baseline": base, "crossing": crossing, "parallel": parallel}
compare_maps(runs); plt.show()
display(compare(runs))

**How to read this**
* **Map**: corridor colour = density at that scenario's busiest moment.
* **time spent inside (person-hours)**: the main "how bad was it" number. **Lower is better.**
* **turned away**: people who could not even enter because the first corridor was full.

**Discuss with your partner**
1. Which scenario takes the most time (person-hours)?
2. Look at corridor 2_3 (Plaza–Hall) on the three maps. Why is it busy in one scenario and almost empty in another?

<details><summary>💡 Our answers</summary>
<b>1.</b> <b>Crossing</b>: about 650 person-hours, against about 435 for baseline and 340 for parallel. The same number
of people reach each destination in all three.<br><br>
<b>2.</b> In crossing, Gate A's walkers go east to the Platform and part of Gate B's walkers go west to the City
Exit, so they meet <b>head-on</b> in the narrow 2 m corridor 2_3 (up to 3.5 people per m²). People walking
towards you count towards the density too, so everyone slows down. In parallel, Gate A heads north and Gate B
heads east: nobody needs 2_3 (0.1 per m²) and the two streams never meet.<br>
So "50% go to the Platform" is not enough to predict the crowding: you need to know <i>who</i> goes where.
</details>

---
## 6 · Same model, a real city
PedNStream does not care how big the network is. This is the network of Part 3: real streets from
**OpenStreetMap** between Amsterdam **Centraal Station** and **Museumplein**, about 120 street segments.
People leave the station at a steady rate for an hour and walk to Museumplein, choosing between 5 routes.
The run takes about 5 seconds.

In [ ]:
#@title ▶ Run Amsterdam {display-mode: "form"}
people_per_minute_from_the_station = 300  #@param {type:"slider", min:60, max:900, step:60}
ams = run(network_dir=AMSTERDAM, arrivals={0: arrival_plan([people_per_minute_from_the_station] * 6)})
ams.dashboard("Centraal Station → Museumplein", flow=True); plt.show()
display(ams.kpis().to_frame("value"))

> **Try:** 300 people per minute, then 600. Does the red line (arriving at Museumplein) get as high as the
> blue (leaving the station)?

<details><summary>💡 Our answer</summary>
300: yes. The first people arrive after about <b>44 minutes</b> (the walk on empty streets), and from then on 300
per minute arrive, as many as leave. 600: no. Arrivals level off at about <b>400</b> per minute: the streets near
Muntplein (Oude Turfmarkt) jam, and when the 2 hours are over about 8,000 people are still on their way.
You'll plan this event yourself in <b>Part 3</b>.
</details>

---
## 7 · Build your own network ⭐
Now you are the modeller. A network needs just a few things:
1. **Nodes**, numbered `0, 1, 2, …` with an **x, y position in metres**. A corridor's length is the
   distance between its two nodes.
2. **Corridors** between pairs of nodes, e.g. `"1-2"`. They are 3 m wide unless you say otherwise.
3. **Origins** (where people enter) and **destinations** (where they want to go).
4. **OD weights** per origin–destination pair, as in sections 4 and 5. Pairs you leave out get weight 0.
5. An **arrival pattern** per origin: people per minute, one value per block of minutes.
   `[120, 240, 120, 0]` with 5-minute blocks = 120/min for 5 minutes, 240/min for 5 minutes, and so on.

The example is a small museum. People enter at the **Entrance** and leave by **Exit East** or **Exit
North**; between them two routes run through the narrow Hall–Square corridor or past the Gallery:
```
                       6 Exit North
                       |
        3 ──────────── 4
     Gallery           |  Garden
        |              |
 0 ──── 1 ════════════ 2 ──── 5
Entrance Hall   (2 m)  Square  Exit East
```
In Colab, type in the boxes on the right (keep the brackets and quotes). In Jupyter, edit the values in the code.

In [ ]:
#@title 🛠️ **Your network**: edit the values, then press ▶ {display-mode: "form"}
#@markdown ### 1 · Nodes: numbers 0, 1, 2, … with x, y in metres, and (optional) names
positions = {0: (0, 0), 1: (60, 0), 2: (140, 0), 3: (60, 70), 4: (140, 70), 5: (200, 0), 6: (140, 130)}  #@param {type:"raw"}
names = {0: "Entrance", 1: "Hall", 2: "Square", 3: "Gallery", 4: "Garden", 5: "Exit East", 6: "Exit North"}  #@param {type:"raw"}
#@markdown ### 2 · Corridors (two-way), and the ones that are not 3 m wide
corridors = ["0-1", "1-2", "1-3", "3-4", "2-4", "2-5", "4-6"]  #@param {type:"raw"}
widths = {"1-2": 2}  #@param {type:"raw"}
#@markdown ### 3 · Where people enter, and where they want to go
origins = [0]  #@param {type:"raw"}
destinations = [5, 6]  #@param {type:"raw"}
#@markdown ### 4 · OD weights, as "origin_destination": weight (pairs you leave out get 0)
od_weights = {"0_5": 2, "0_6": 1}  #@param {type:"raw"}
#@markdown ### 5 · Arrival pattern: people per minute at each origin, one value per block
arrivals_per_minute = {0: [120, 240, 120, 0]}  #@param {type:"raw"}
block_minutes = 5  #@param {type:"slider", min:1, max:15, step:1}
simulate_minutes = 40  #@param {type:"slider", min:10, max:120, step:5}

folder = make_network(corridors, positions, origins, destinations, widths=widths, names=names,
                      folder="my_network", minutes=simulate_minutes)
od = {f"{o}_{d}": od_weights.get(f"{o}_{d}", 0) for o in origins for d in destinations}
for o in origins:
    if o not in arrivals_per_minute:
        raise ValueError(f"Origin {o} has no arrival pattern: add it to arrivals_per_minute")
    if not any(od[f"{o}_{d}"] for d in destinations):
        raise ValueError(f"Origin {o}: give at least one OD weight above 0, e.g. '{o}_{destinations[0]}': 1")
arrivals = {o: arrival_plan(plan, block_minutes) for o, plan in arrivals_per_minute.items()}
mine = run(network_dir=folder, od=od, arrivals=arrivals)
mine.dashboard("My network", flow=True); plt.show()
display(mine.kpis().to_frame("value"))
print("Which routes people chose (share per OD pair, at the junction where the routes split):")
display(mine.route_split())

**Ideas to try**
1. **Same people, different timing.** Try arrivals `{0: [480, 0, 0, 0]}`, then `{0: [120, 120, 120, 120]}`.
   Both bring 2,400 people. Which one is better?
2. **Make a route narrow.** Set `widths = {"1-2": 2, "1-3": 1}`. Which route do people take now?
3. **A second entrance.** Add a Side Gate north of the Gallery:
   * `positions`: add `7: (60, 130)`, and `names`: add `7: "Side Gate"`
   * `corridors`: add `"3-7"`, and set `origins = [0, 7]`
   * `arrivals_per_minute`: add `7: [120, 240, 120, 0]`

   Then compare `od_weights = {"0_6": 1, "7_5": 1}` with `{"0_5": 1, "7_6": 1}`. Which one is worse, and why?
4. **Your own place.** Sketch a station, a stadium or your campus on paper, number the junctions,
   guess their positions in metres, and type it in. Start with one origin and one destination.

<details><summary>💡 Our answers</summary>
<b>1.</b> The burst <code>[480, 0, 0, 0]</code> <i>looks</i> better (131 person-hours against 161), but look at
<i>turned away</i>: about 700 of the 2,400 people could not even get in, so their time is not counted. Spread out,
nobody is turned away. Always check <i>turned away</i> next to the time.<br><br>
<b>2.</b> Almost everyone switches to the Hall–Square corridor 1_2: from 66% to 97% of the people going to Exit East,
and from 27% to 84% of those going to Exit North. Corridor 1_2 climbs from 1.0 to almost 2 people per m², right at
the edge of slowing down. So the crowd moves to the other route; with more people, 1_2 would jam.<br><br>
<b>3.</b> <code>{"0_6": 1, "7_5": 1}</code> is worse: about 330 person-hours against 280. Entrance → Exit North walks
north and east, Side Gate → Exit East walks east and south, so the two streams meet <b>head-on</b> in corridors
1_3 and 2_4 and share 3_4. With <code>{"0_5": 1, "7_6": 1}</code> they hardly meet. Same idea as section 5: who goes
where decides where it gets crowded.
</details>

In [ ]:
#@title 🎬 Animate your network (optional, ≈15 s) {display-mode: "form"}
display(mine.animate(every=3))

### Your network is a real PedNStream network
`make_network` wrote a folder `my_network/` with a `sim_params.yaml`, the file PedNStream reads. Everything
you set above is in it, plus defaults you can change (walking speed, densities, route-choice settings).
In **Part 4** you'll write such a file yourself and run PedNStream directly.

In [ ]:
#@title ▶ Show my sim_params.yaml {display-mode: "form"}
print(open("my_network/sim_params.yaml").read())

---
## Take-away
* PedNStream tracks how many people are on each corridor. Every 10 s people **arrive** (arrival pattern),
  get a **destination** (OD weights), pick a **route** (distance, width, crowding) and **walk** at a speed
  set by the density.
* A corridor can pass only so many people per minute. Above that a **queue** forms and grows **backwards**.
* An OD weight sets **where** people want to go, not **how many** there are. Two OD patterns with the same
  totals can behave very differently: when routes **overlap** and walkers meet **head-on**, corridors jam.
* The same model runs a single corridor, a station and a city centre, and any network you can sketch.

➡ Next: **Part 2 · Playground**: change the OD weights, the demand, how much people avoid crowds and
the width of corridor 2_3 yourself. Then **Part 3 · Challenge**.